In [0]:
%pip install --upgrade pydantic



Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
# Descarga dataset de ejemplo
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import mlflow
import mlflow.sklearn

# Carga y prepara datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

# Define hiperparámetros para experimentar
param_grid = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9}
]

mlflow.set_experiment("/Users/guillermo.henrion@gmail.com/Itba 2026/RandomForest_BreastCancer")

for params in param_grid:
    with mlflow.start_run():
        clf = RandomForestClassifier(n_estimators=params["n_estimators"], max_depth=params["max_depth"], random_state=42)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]
        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        #mlflow.sklearn.log_model(clf, "model")

In [0]:
mlflow.sklearn.log_model(clf, name="model")

🔗 View Logged Model at: https://dbc-b9e8bb32-1d55.cloud.databricks.com/ml/experiments/3343515417927605/models/m-c67792e72b7a4ea79f1585039d9b90c4?o=273782750543210
2026/03/19 00:58:18 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.10.1/ml/model/signatures.html for instructions on setting signature on models.


In [0]:
# Descarga dataset de ejemplo
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import pydantic

# Carga y prepara datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

# Define hiperparámetros para experimentar
param_grid = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9}
]

mlflow.set_experiment("/Users/guillermo.henrion@gmail.com/Itba 2026/RandomForest_BreastCancer")

best_auc = 0
best_run_id = None

for params in param_grid:
    with mlflow.start_run() as run:
        clf = RandomForestClassifier(n_estimators=params["n_estimators"], max_depth=params["max_depth"], random_state=42)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]
        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        cm = confusion_matrix(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True)
        signature = infer_signature(X_test, y_pred)
        
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("precision", report["1"]["precision"])
        mlflow.log_metric("recall", report["1"]["recall"])
        mlflow.log_metric("f1_score", report["1"]["f1-score"])
        mlflow.sklearn.log_model(clf, signature=signature)
        mlflow.log_dict(report, "classification_report.json")
        mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")
        
        if auc > best_auc:
            best_auc = auc
            best_run_id = run.info.run_id

# Publicar el mejor modelo
if best_run_id:
    mlflow.register_model(f"runs:/{best_run_id}/model", "RandomForest_BreastCancer_Best")

🔗 View Logged Model at: https://dbc-b9e8bb32-1d55.cloud.databricks.com/ml/experiments/3343515417927605/models/m-f5c60f84c7a84e288f54494965329774?o=273782750543210
🔗 View Logged Model at: https://dbc-b9e8bb32-1d55.cloud.databricks.com/ml/experiments/3343515417927605/models/m-f1d8451d697f4116bb87a6bb0aa2f956?o=273782750543210
🔗 View Logged Model at: https://dbc-b9e8bb32-1d55.cloud.databricks.com/ml/experiments/3343515417927605/models/m-977c4770fb2b4e38a4a7a8887e32c3f9?o=273782750543210
🔗 View Logged Model at: https://dbc-b9e8bb32-1d55.cloud.databricks.com/ml/experiments/3343515417927605/models/m-9850a2f627c54bfcbd0d13f987aed37b?o=273782750543210
Registered model 'RandomForest_BreastCancer_Best' already exists. Creating a new version of this model...


---------------------------------------------------------------------------
MlflowException                           Traceback (most recent call last)
File <command-5827969030465066>, line 62
     60 # Publicar el mejor modelo
     61 if best_run_id:
---> 62     mlflow.register_model(f"runs:/{best_run_id}/model", "RandomForest_BreastCancer_Best")

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-aa89ab41-d6fc-4d1a-8ba6-3f15b20c0432/lib/python3.12/site-packages/mlflow/tracking/_model_registry/fluent.py:148, in register_model(model_uri, name, await_registration_for, tags, env_pack)
     67 def register_model(
     68     model_uri,
     69     name,
   (...)
     73     env_pack: EnvPackType | EnvPackConfig | None = None,
     74 ) -> ModelVersion:
     75     """Create a new model version in model registry for the model files specified by ``model_uri``.
     76 
     77     Note that this method assumes the model registry backend URI is the same as that of the
   (...)
    146         V

In [0]:
%pip install evidently

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from evidently import Report
from evidently.metric_preset import DataDriftPreset, ClassificationPreset

# Evidently analysis for the last trained model
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

report = Report(metrics=[
    ClassificationPreset(),
    ClassificationPreset()
])

report.run(
    reference_data=pd.DataFrame(X_train).assign(target=y_train),
    current_data=pd.DataFrame(X_test).assign(target=y_test, prediction=y_pred, proba=y_proba)
)

report.show()

---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
File <command-5827969030465068>, line 2
      1 from evidently import Report
----> 2 from evidently.metric_preset import BaseResult, DataDriftPreset, ClassificationPreset
      4 # Evidently analysis for the last trained model
      5 y_pred = clf.predict(X_test)

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-14a06dec-5528-44fa-93a4-0eb22166f0d3/lib/python3.12/site-packages/evidently/metric_preset/__init__.py:1
----> 1 from .classification_performance import ClassificationPreset
      2 from .data_drift import DataDriftPreset
      3 from .data_quality import DataQualityPreset

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-14a06dec-5528-44fa-93a4-0eb22166f0d3/lib/python3.12/site-packages/evidently/metric_preset/classification_performance.py:4
      1 from typing import List
      2 from typing import Optional
----> 4 from evidently.